# Partition 1 EDA

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Extracting partition 1

In [ ]:
import os

# Where the archive currently lives
drive_data_dir = '/content/drive/MyDrive/solar_flare_forecasting/Data'
# True high-speed local Colab SSD path
local_extract_dir = '/content/solar_flare_data'

# Ensure the local extraction directory exists
os.makedirs(local_extract_dir, exist_ok=True)

files_to_extract = {
    "partition1_instances.tar.gz": "https://dataverse.harvard.edu/api/access/datafile/:persistentId?persistentId=doi:10.7910/DVN/EBCFKM/BMXYCB"
}

print("\n--- Starting High-Speed Local Extraction ---")

for file_name in files_to_extract.keys():
    archive_path = os.path.join(drive_data_dir, file_name)

    if os.path.exists(archive_path):
        print(f"⚡ Unpacking {file_name} into Colab SSD ({local_extract_dir})...")
        # -xzf: extract gzipped file, -C: target local directory
        # Using native system tar is indeed the fastest method
        exit_code = os.system(f"tar -xzf {archive_path} -C {local_extract_dir}")

        if exit_code == 0:
            print(f"✅ Successfully unpacked {file_name}!")
        else:
            print(f"❌ Error occurred while unpacking {file_name}. Exit code: {exit_code}")
    else:
        print(f"⚠️ Could not find archive at {archive_path}, skipping extraction.")

print(f"\n🎉 All processes complete! Data is ready in '{local_extract_dir}'.")

## Exploring all files in Partition 1.

for missing data and time stamp differences


### Exploration loop

In [ ]:
import pandas as pd
import numpy as np
import glob
from collections import Counter

fl_files = glob.glob("/content/solar_flare_data/partition1/FL/*.csv")
nf_files = glob.glob("/content/solar_flare_data/partition1/NF/*.csv")
all_files = fl_files + nf_files

print(f"Total files found in Partition 1: {len(all_files)} ({len(fl_files)} FL, {len(nf_files)} NF)")

# Reset metrics
total_rows_checked = 0
total_missing_cells = 0
total_missing_rows = 0
files_with_missing_values = 0
cadence_differences = Counter()
unexpected_row_counts = []

# --- CONFIGURATION ---
SAMPLE_LIMIT = 200
files_to_process = all_files[:SAMPLE_LIMIT] if SAMPLE_LIMIT else all_files

print(f"Processing {len(files_to_process)} files...")

for path in files_to_process:
    df = pd.read_csv(path, sep='\t')

    num_rows = len(df)
    total_rows_checked += num_rows

    if num_rows != 60:
        unexpected_row_counts.append((path.split('/')[-1], num_rows))

    null_cells = df.isnull().sum().sum()
    if null_cells > 0:
        total_missing_cells += null_cells
        files_with_missing_values += 1
        total_missing_rows += df.isnull().any(axis=1).sum()

    ts_col = [c for c in df.columns if 'timestamp' in c.lower()]
    if ts_col:
        try:
            # FIX: Specified format to remove UserWarning and speed up parsing
            timestamps = pd.to_datetime(df[ts_col[0]], format='%Y-%m-%d %H:%M:%S', errors='coerce')
            time_deltas = timestamps.diff().dropna()
            delta_minutes = time_deltas.dt.total_seconds() / 60.0
            for delta in delta_minutes:
                cadence_differences[delta] += 1
        except Exception as e:
            pass

print("\n--- EXPLORATION COMPLETED ---")

### Report

In [ ]:
print("==================================================")
print("             DATA QUALITY ANALYSIS REPORT         ")
print("==================================================")

# Calculations
missing_cell_pct = (total_missing_cells / (total_rows_checked * 55)) * 100 if total_rows_checked > 0 else 0
missing_row_pct = (total_missing_rows / total_rows_checked) * 100 if total_rows_checked > 0 else 0

# 1. Missing Value Report
print(f"• Total Rows Scanned:      {total_rows_checked:,}")
print(f"• Files with Null Values:   {files_with_missing_values} out of {len(files_to_process)}")
print(f"• Total Missing Data Cells: {total_missing_cells:,} ({missing_cell_pct:.2f}% of all individual cells)")
print(f"• Total Rows with Missing:  {total_missing_rows:,} ({missing_row_pct:.2f}% of all scanned rows) ⚠️")

print("\n--------------------------------------------------")
print("• Row Count Deviations (Expected: 60 rows per file):")
if len(unexpected_row_counts) == 0:
    print("  -> Excellent! Every scanned file has exactly 60 rows.")
else:
    print(f"  -> Found {len(unexpected_row_counts)} files with non-60 row limits:")
    for file_name, rows in unexpected_row_counts[:10]:
        print(f"     - {file_name}: {rows} rows")

print("\n--------------------------------------------------")
print("• Timestamp Difference / Cadence Distribution:")
if not cadence_differences:
    print("  -> No timestamp cadence could be computed.")
else:
    print("  Expected step size is 12.0 minutes.")
    for minutes, count in sorted(cadence_differences.items()):
        status = " (Expected Cadence)" if minutes == 12.0 else " ⚠️ (Cadence Gap/Jump)"
        print(f"     - Gap of {minutes} minutes: {count:,} occurrences{status}")
print("==================================================")

In [ ]:
import pandas as pd

try:
    # Extracting and displaying the QUALITY counts from the quality_stats dictionary
    quality_counts = pd.Series(quality_stats['QUALITY']).sort_index()
    print("Value counts for the 'QUALITY' feature (Current Sample):")
    display(quality_counts)
except NameError:
    print("❌ Error: 'quality_stats' not found.")
    print("Please run the 'Value count loop' cell (QKmI_TP4WZ7w) above to populate the quality data first.")

### Exploration excluding the Label columns

`BFLARE_LABEL`,
 `CFLARE_LABEL`,
 `MFLARE_LABEL`,
 `XFLARE_LABEL`,
 `BFLARE_LABEL_LOC`,
 `CFLARE_LABEL_LOC`,
 `MFLARE_LABEL_LOC`,
 `XFLARE_LABEL_LOC`

 They are empty most of the time as Flares are the minority class.

In [ ]:
import pandas as pd
import numpy as np
import glob
from collections import Counter

COLS_TO_EXCLUDE = [
    'BFLARE_LABEL', 'CFLARE_LABEL', 'MFLARE_LABEL', 'XFLARE_LABEL',
    'BFLARE_LABEL_LOC', 'CFLARE_LABEL_LOC', 'MFLARE_LABEL_LOC', 'XFLARE_LABEL_LOC'
]

fl_files = glob.glob("/content/solar_flare_data/partition1/FL/*.csv")
nf_files = glob.glob("/content/solar_flare_data/partition1/NF/*.csv")
all_files = fl_files + nf_files

# Reset metrics
total_rows_checked = 0
total_missing_cells = 0
total_missing_rows = 0
files_with_missing_values = 0
cadence_differences = Counter()
unexpected_row_counts = []

SAMPLE_LIMIT = 500
files_to_process = all_files[:SAMPLE_LIMIT] if SAMPLE_LIMIT else all_files

print(f"Processing {len(files_to_process)} files (Excluding label columns)...")

for path in files_to_process:
    df = pd.read_csv(path, sep='\t')
    df_features = df.drop(columns=[c for c in COLS_TO_EXCLUDE if c in df.columns])

    num_rows = len(df_features)
    total_rows_checked += num_rows

    if num_rows != 60:
        unexpected_row_counts.append((path.split('/')[-1], num_rows))

    null_cells = df_features.isnull().sum().sum()
    if null_cells > 0:
        total_missing_cells += null_cells
        files_with_missing_values += 1
        total_missing_rows += df_features.isnull().any(axis=1).sum()

    ts_col = [c for c in df.columns if 'timestamp' in c.lower()]
    if ts_col:
        # FIX: Specified format to remove UserWarning and speed up parsing
        timestamps = pd.to_datetime(df[ts_col[0]], format='%Y-%m-%d %H:%M:%S', errors='coerce')
        time_deltas = timestamps.diff().dropna()
        delta_minutes = time_deltas.dt.total_seconds() / 60.0
        for delta in delta_minutes:
            cadence_differences[delta] += 1

num_features_checked = df_features.shape[1]
print(f"\n--- EXPLORATION COMPLETED ({num_features_checked} features analyzed) ---")

### Report

Excluding Sparse Label Columns

In [ ]:
print("==================================================")
print("        DATA QUALITY ANALYSIS REPORT       ")
print("      (Excluding Sparse Label Columns)           ")
print("==================================================")

# Dynamic feature count from previous cell
features_count = num_features_checked if 'num_features_checked' in locals() else 47

# Calculations
missing_cell_pct = (total_missing_cells / (total_rows_checked * features_count)) * 100 if total_rows_checked > 0 else 0
missing_row_pct = (total_missing_rows / total_rows_checked) * 100 if total_rows_checked > 0 else 0

# 1. Missing Value Report
print(f"• Total Rows Scanned:      {total_rows_checked:,}")
print(f"• Features per Row:        {features_count}")
print(f"• Files with Null Values:   {files_with_missing_values} out of {len(files_to_process)}")
print(f"• Total Missing Data Cells: {total_missing_cells:,} ({missing_cell_pct:.4f}% of total cells)")
print(f"• Total Rows with Missing:  {total_missing_rows:,} ({missing_row_pct:.2f}% of scanned rows)")

print("\n--------------------------------------------------")
print("• Row Count Deviations (Expected: 60 rows):")
if not unexpected_row_counts:
    print("  -> All files have exactly 60 rows.")
else:
    print(f"  -> Found {len(unexpected_row_counts)} anomalies.")

print("\n--------------------------------------------------")
print("• Cadence Distribution:")
for minutes, count in sorted(cadence_differences.items()):
    status = " (Standard)" if minutes == 12.0 else " ⚠️ (Irregular)"
    print(f"     - {minutes} min: {count:,} times{status}")
print("==================================================")

In [ ]:
missing_rows_list = []
files_affected = []

for path in files_to_process:
    df = pd.read_csv(path, sep='\t')
    df_features = df.drop(columns=[c for c in COLS_TO_EXCLUDE if c in df.columns])

    mask = df_features.isnull().any(axis=1)
    if mask.any():
        filename = path.split('/')[-1]
        files_affected.append(filename)
        rows = df_features[mask].copy()
        rows['source_file'] = filename
        missing_rows_list.append(rows)

if missing_rows_list:
    all_missing_df = pd.concat(missing_rows_list, ignore_index=True)
    print(f"✅ Total rows with missing data: {len(all_missing_df)}")
    print(f"📂 Affected files ({len(files_affected)} total): {files_affected}")

    # FIX: Arguments must be in pairs (pattern, value)
    display(all_missing_df)
else:
    print("No missing data found in the current sample subset.")

In [ ]:
all_missing_df

## Analyzing Quality of rows

Based on data quality parameters:

`QUALITY`, `IS_TMFI`, `SPEI`, `XRQUALITY`

| Column     | Meaning |
|------------|---------|
| `QUALITY`  | Bitmask flag from HMI pipeline. **0 = good data**. Non-zero = something went wrong. |
| `IS_TMFI`  | "Trusted Magnetic Field Information" boolean. **`True` = trustworthy** (`QUALITY = 0` and within 70° of disk center). |
| `SPEI`     | Boolean — **`True`** means SDO was in Earth's shadow (data blackout); exclude these rows. |
| `XRQUALITY`| Quality flag for X-ray flux data — indicates how many of the 1-minute readings in the window were valid. |

### Value count loop

In [ ]:
from collections import Counter

# Initialize counters for each quality column
quality_stats = {
    'QUALITY': Counter(),
    'IS_TMFI': Counter(),
    'SPEI': Counter(),
    'XR_QUAL': Counter()
}

print(f"Analyzing quality flags across {len(files_to_process)} files...")

for path in files_to_process:
    df = pd.read_csv(path, sep='\t')

    for col in quality_stats.keys():
        if col in df.columns:
            # Update counter with values from this file
            quality_stats[col].update(df[col].fillna('NaN').tolist())
        else:
            # Track missing columns if they don't exist in some files
            quality_stats[col]['COLUMN_MISSING'] += len(df)

### Report

In [ ]:
print("==================================================")
print("         GLOBAL QUALITY FLAG REPORT              ")
print("==================================================")

def print_stat_group(name, counter_dict, description):
    print(f"\n• {name}:")
    print(f"  ({description})")
    # Sort keys by string to ensure NaN and numbers don't conflict
    sorted_keys = sorted(counter_dict.keys(), key=lambda x: str(x))
    for key in sorted_keys:
        count = counter_dict[key]
        pct = (count / total_rows_checked) * 100
        print(f"    - {key}: {count:,} rows ({pct:.2f}%)")

# 1. HMI Quality Bitmask
print_stat_group(
    "HMI QUALITY FLAGS",
    quality_stats['QUALITY'],
    "0 = Good data, Non-zero = Error bitmask"
)

# 2. Trusted Magnetic Field Information
print_stat_group(
    "IS_TMFI (Trusted Magnetic Field)",
    quality_stats['IS_TMFI'],
    "True = Trustworthy, False = Potential issues"
)

# 3. SDO Earth Shadow (SPEI)
print_stat_group(
    "SPEI (Earth Shadow Indicator)",
    quality_stats['SPEI'],
    "True = Normal, False = SDO in Earth's shadow (Blackout)"
)

# 4. X-Ray Quality
print_stat_group(
    "XR_QUAL (X-Ray Flux Quality)",
    quality_stats['XR_QUAL'],
    "Count of valid 1-min readings (12 = Perfect)"
)

print("\n==================================================")

## Making dataframe with rows that "fail" the GLOBAL QUALITY FLAG REPORT

`QUALITY` != 0.0 <br>
`IS_TMFI` == false <br>
`SPEI` == false <br>
`XR_QUAL` != 12 <br>

In [ ]:
any_bad_quality_rows = []

# Using the sample of 200 files
for path in files_to_process:
    df_temp = pd.read_csv(path, sep='\t')

    # Use OR (|) to find rows where ANY condition is met
    mask = (
        (df_temp['QUALITY'] != 0.0) |
        (df_temp['IS_TMFI'] == False) |
        (df_temp['SPEI'] == False) |
        (df_temp['XR_QUAL'] != 12)
    )

    if mask.any():
        match = df_temp[mask].copy()
        match['source_file'] = path.split('/')[-1]
        any_bad_quality_rows.append(match)

if any_bad_quality_rows:
    any_worst_df = pd.concat(any_bad_quality_rows, ignore_index=True)
    print(f"Found {len(any_worst_df)} rows meeting at least one 'bad quality' criterion.")
    #display(any_worst_df.head())
else:
    print("No rows in the sample met any of the bad quality criteria.")

In [ ]:
any_worst_df

Check if the  NaN-quality rows are exactly the SPEI=False rows

In [ ]:
# Filter the previously combined missing data dataframe
nan_quality_rows = any_worst_df[any_worst_df['QUALITY'].isna()]
spei_false_rows = any_worst_df[any_worst_df['SPEI'] == False]

# Compare lengths and indices
match_count = len(nan_quality_rows) == len(spei_false_rows)

print(f"Number of NaN Quality rows: {len(nan_quality_rows)}")
print(f"Number of SPEI=False rows: {len(spei_false_rows)}")
print(f"Do the indices match exactly? {match_count}")

if match_count:
    print("\nConfirmation: The rows with missing physical features are a 100% match for SDO Earth Shadow events.")

In [ ]:
import pandas as pd

file_path = '/content/solar_flare_data/partition1/FL/M1.0@1235:Primary_ar384_s2011-02-16T23:00:00_e2011-02-17T10:48:00.csv'

try:
    # Reading only the header to get column names efficiently
    df_cols = pd.read_csv(file_path, sep='\t', nrows=0)
    print(f"Columns in file: {file_path}\n")
    print(df_cols.columns.tolist())
except FileNotFoundError:
    print(f"❌ Error: The file {file_path} was not found.")